# Pipeline 7 : Dessiner la carte de l'espace chimique

La question du chercheur : peut-on voir d'un seul coup d'oeil ou se trouvent les molecules actives dans l'immense espace des structures possibles ?

Une molecule vit dans un espace a des milliers de dimensions, celui de ses empreintes structurelles. Aucun humain ne peut se representer un tel espace. La reduction de dimension projette cet espace sur un plan a deux dimensions, en essayant de preserver au mieux les proximites : deux molecules structurellement proches doivent rester proches sur la carte. On obtient ainsi une carte de l'espace chimique de l'EGFR, sur laquelle on peut voir emerger les continents des molecules actives.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import joblib

C_BLEU = "#1f6f8b"
C_ORANGE = "#e0771a"
C_VERT = "#2e8b57"
C_ROUGE = "#9b2226"
C_GRIS = "#8d99ae"

plt.rcParams.update({
    "figure.figsize": (13, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 13
})

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('egfr_descripteurs.csv')
fingerprints = np.load('egfr_fingerprints.npy')
df['activite'] = np.where(df['pIC50'] >= 6, 'actif',
                          np.where(df['pIC50'] < 5, 'inactif', 'intermediaire'))
print(f"Molecules : {len(df)}")

# 1. La PCA : une carte lineaire et interpretable

La PCA cherche les directions qui capturent le plus de variance dans les donnees. Son avantage est qu'elle est lineaire et donc interpretable, son inconvenient est qu'elle ecrase souvent les structures complexes. On regarde d'abord combien de variance elle recupere.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords_pca = pca.fit_transform(fingerprints)
df['pca1'], df['pca2'] = coords_pca[:, 0], coords_pca[:, 1]
var = pca.explained_variance_ratio_ * 100

plt.figure(figsize=(13, 8))
for etat, couleur, taille in [('inactif', C_GRIS, 18), ('actif', C_VERT, 30)]:
    sous = df[df['activite'] == etat]
    plt.scatter(sous['pca1'], sous['pca2'], s=taille, alpha=0.5,
                color=couleur, label=etat, edgecolors='none')
plt.xlabel(f"Composante 1 ({var[0]:.1f}% de variance)")
plt.ylabel(f"Composante 2 ({var[1]:.1f}% de variance)")
plt.title("Espace chimique par PCA, colore par activite")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Les deux premieres composantes ne capturent que {var.sum():.0f}% de la variance, ce qui est peu. C'est normal et instructif : les empreintes moleculaires sont tres creuses et tres riches, l'information ne se comprime pas bien en deux directions lineaires. On distingue une certaine tendance des molecules actives a se regrouper, mais la separation reste floue. La PCA atteint ici sa limite, ce qui justifie de passer a une methode non lineaire.")

# 2. Le t-SNE : le graphique signature

Le t-SNE est une methode non lineaire concue pour preserver les voisinages locaux. Elle est bien plus efficace que la PCA pour reveler des groupes, au prix de deux limites : elle est lente et ses axes n'ont pas de signification directe. On l'applique sur une PCA intermediaire a 30 composantes, une pratique recommandee qui accelere le calcul et reduit le bruit.

In [ ]:
# Reduction intermediaire pour accelerer et debruiter le t-SNE
pca_inter = PCA(n_components=30, random_state=RANDOM_STATE)
X_inter = pca_inter.fit_transform(fingerprints)

tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, init='pca')
coords_tsne = tsne.fit_transform(X_inter)
df['tsne1'], df['tsne2'] = coords_tsne[:, 0], coords_tsne[:, 1]
gc.collect()

plt.figure(figsize=(14, 9))
sc = plt.scatter(df['tsne1'], df['tsne2'], c=df['pIC50'], cmap='viridis',
                 s=30, alpha=0.7, edgecolors='none')
plt.colorbar(sc, label='pIC50 (puissance)')
plt.xlabel("Dimension t-SNE 1")
plt.ylabel("Dimension t-SNE 2")
plt.title("La carte de l'espace chimique de l'EGFR (t-SNE), coloree par puissance")
plt.tight_layout()
plt.show()

print("Voila la carte que l'equipe accrochera au mur. Chaque point est une molecule, et les molecules structurellement proches se retrouvent cote a cote. La coloration par puissance revele quelque chose que la PCA ne montrait pas : des ilots de molecules puissantes, en jaune, se detachent nettement de l'ocean des molecules faibles. Ces ilots sont les series chimiques prometteuses. Un chercheur qui a une nouvelle molecule peut la projeter sur cette carte et voir immediatement si elle tombe dans un ilot actif ou dans une zone morte.")

In [ ]:
# La meme carte, mais coloree par famille chimique si le notebook 6 a ete execute
try:
    familles = pd.read_csv('egfr_familles.csv')
    df = df.merge(familles[['canonical_smiles', 'famille']], on='canonical_smiles', how='left')
    PALETTE = [C_BLEU, C_ORANGE, C_VERT, C_ROUGE, "#5f0f40", "#d4a017", "#3a86ff"]
    plt.figure(figsize=(14, 9))
    for fam in sorted(df['famille'].dropna().unique()):
        sous = df[df['famille'] == fam]
        plt.scatter(sous['tsne1'], sous['tsne2'], s=28, alpha=0.6,
                    color=PALETTE[int(fam) % len(PALETTE)], label=f"Famille {int(fam)}", edgecolors='none')
    plt.xlabel("Dimension t-SNE 1")
    plt.ylabel("Dimension t-SNE 2")
    plt.title("La meme carte, coloree par famille chimique du clustering")
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("En superposant les familles trouvees par le clustering sur la carte t-SNE, on verifie visuellement la coherence des deux approches. Si les familles forment des zones bien delimitees sur la carte, c'est que le clustering et le t-SNE, deux methodes independantes, s'accordent sur la structure des donnees, ce qui renforce notre confiance dans les deux.")
except FileNotFoundError:
    print("Executer d'abord le notebook 6 pour disposer des familles chimiques et colorer la carte par famille.")

# 3. Deploiement

In [ ]:
joblib.dump({'pca_2d': pca, 'pca_inter': pca_inter}, 'modele_projection.pkl')
# On sauvegarde les coordonnees de la carte pour le dashboard
df[['canonical_smiles', 'pIC50', 'activite', 'tsne1', 'tsne2', 'pca1', 'pca2']].to_csv(
    'egfr_carte.csv', index=False)
print("Sauvegarde : modele_projection.pkl et egfr_carte.csv")
print("Le dashboard positionnera les nouvelles molecules sur cette carte de reference.")

# Conclusion

La comparaison entre PCA et t-SNE est un enseignement en soi. La PCA, lineaire et interpretable, peine a resumer un espace aussi riche que celui des empreintes moleculaires. Le t-SNE, non lineaire, revele des ilots de molecules actives invisibles autrement, et livre la carte la plus parlante de tout le projet.

Il faut cependant manier le t-SNE avec prudence, et on l'assume. Ses axes n'ont aucune signification, seules les proximites locales comptent, et les distances entre groupes eloignes ne sont pas fiables. On ne peut donc pas dire qu'un ilot est deux fois plus loin qu'un autre. La carte sert a explorer et a situer, pas a mesurer. Utilisee ainsi, elle est un formidable outil d'orientation pour la recherche.